# Building a GAN From Scratch

**No PyTorch. No autograd. No GPU.**

We will build a small GAN in NumPy and train it to generate 8x8 handwritten digits.

1. Load the data
2. Build the neural network
3. Work out the GAN gradients
4. Train the discriminator and generator
5. Inspect the results

---
## 1. The data

Each image is 8x8 pixels, so each example becomes a vector of 64 numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_digits

SEED = 0
rng = np.random.default_rng(SEED)

digits = load_digits()

# Our neural networks will receive each image as one vector rather than as an
# 8x8 grid, so reshape converts every image into a row of 64 pixel values

# The original pixel intensities range from 0 to 16. We rescale them to [-1, 1]
# because the generator's final tanh activation will also produce values in
# that range
X = digits.images.reshape(-1, 64) / 8.0 - 1.0
y = digits.target
IMG = (8, 8)

print(
    f"{X.shape[0]} images, "
    f"{X.shape[1]} pixels each, "
    f"range [{X.min():.1f}, {X.max():.1f}]"
)

In [ ]:
def show_grid(flat, title="", n=64, ncol=8, scale=0.7, labels=None):
  """Display flattened digit images in a grid."""
    # Do not request more images than the supplied collection contains
    n = min(n, len(flat))

    # Determine how many rows are needed to display n images using ncol columns
    # np.ceil ensures that a partially filled final row still receives space
    nrow = int(np.ceil(n / ncol))

    # Create the complete grid of plotting areas
    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(ncol * scale, nrow * scale + 0.5)
    )

    # plt.subplots may return a single Axes object, a 1D array, or a 2D array,
    # depending on the grid dimensions
    # Converting to a NumPy array and
    # flattening it gives us one consistent list of plotting areas
    axes = np.array(axes).ravel()

    # Pair each plotting area with one flattened image
    for j, (ax, im) in enumerate(zip(axes, flat[:n])):
        ax.imshow(im.reshape(IMG), cmap="gray", vmin=-1, vmax=1)

        if labels is not None:
            ax.set_title(str(labels[j]), fontsize=7, pad=1)

        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    if title:
        fig.suptitle(title, fontsize=10)

    plt.tight_layout()
    plt.show()

# Look at examples from the real training data
sample_indices = rng.integers(0, len(X), 64)
show_grid(X[sample_indices], "REAL 8x8 digits")

---
## 2. Build the neural network

- **Generator:** noise $\rightarrow$ 64 pixel values
- **Discriminator:** 64 pixel values $\rightarrow$ one logit

The important detail: `backward()` must return the gradient with respect to the network's **input** so the generator can learn through the discriminator.

In [ ]:
def lrelu(a, slope=0.2):
    pass


def dlrelu(a, slope=0.2):
    pass


def sigmoid(z):
    pass


class MLP:
  pass

---
## 3. The GAN gradients

Let $D(x)=\sigma(u)$, where $u$ is the discriminator's logit.

### Discriminator

$$
\mathcal{L}_D
=
-\log D(x)
-
\log\left(1-D(G(z))\right)
$$

$$
\frac{\partial \mathcal{L}_D}{\partial u_{\text{real}}}
=
D(x)-1
$$

$$
\frac{\partial \mathcal{L}_D}{\partial u_{\text{fake}}}
=
D(G(z))
$$

### Generator

We use the non-saturating objective:

$$
\mathcal{L}_G=-\log D(G(z))
$$

so

$$
\frac{\partial \mathcal{L}_G}{\partial u_{\text{fake}}}
=
D(G(z))-1
$$

In [ ]:
class Adam:
    pass

---
## 4. Training

Train the discriminator first on real and fake images. Then train the generator by sending its gradient backward through the discriminator.

In [ ]:
STEPS = 6000
NOISE = 16
HID = 256
BATCH = 128

LR_G = 3e-4
LR_D = 1e-4
SMOOTH = 0.9


def train_gan(
    steps=STEPS,
    lr_g=LR_G,
    lr_d=LR_D,
    noise=NOISE,
    seed=0,
    snapshot_every=1000,
    verbose=True
):
    pass

---
## 5. Did it work?

After training, compare fresh generated images with real examples from the dataset.

In [ ]:
# Compare generated images with real images

# Create 64 random latent vectors
# Each row is one independent starting point in the generator's 16-dimensional
# latent space

# shape = (number of images, latent dimensions) = (64, NOISE)
# These values are sampled from a standard normal distribution, meaning they
# are centered around 0 with a standard deviation of 1
z = rng.standard_normal((64, NOISE))

# Pass the random vectors through the trained generator
# The generator transforms each 16-value latent vector into a 64-value image
# (64, 16) -> G -> (64, 64)
# show_grid() then reshapes each 64-value output into an 8x8 image
show_grid(
    G.forward(z),
    "GENERATED"
)

# Displaying real and generated samples separately lets us compare their
# overall structure, sharpness, variety, and resemblance to handwritten digits
real_indices = rng.integers(0, len(X), 64)

# View real images for comparison
show_grid(
    X[real_indices],
    "REAL"
)

In [ ]:
# Visualize how the generator changed during training

# snapshots contains pairs in this form: (training_step, 16_generated_images)
# Instead of displaying every saved snapshot, select snapshots at roughly five
# evenly spaced intervals
# This keeps the output manageable when many snapshots were recorded
# max(1, ...) prevents a slice step of zero when fewer than five snapshots exist
snapshot_spacing = max(1, len(snapshots) // 5)

for step, images in snapshots[::snapshot_spacing]:
    # Display all 16 images in a single row
    # Read the figures like a training flipbook: early generator outputs should
    # appear mostly formless, while later outputs should develop increasingly
    # recognizable strokes and digit-like structure
    show_grid(
        images,
        title=f"step {step}",
        n=16,
        ncol=16,
        scale=0.75
    )